In [12]:
import pandas as pd
import numpy as np

# ==============================
# 1. Cargar archivo
# ==============================

file_path = "DatosAutomoviles.xlsx"
xls = pd.ExcelFile(file_path)

estado = pd.read_excel(xls, "EstadoGeneral2025")

marcas_2025_h = pd.read_excel(xls, "marcas2025Hibridos")
marcas_2025_e = pd.read_excel(xls, "marcas2025Electricos")
marcas_2023_h = pd.read_excel(xls, "marcas2023Hibridos")
marcas_2023_e = pd.read_excel(xls, "marcas2023Electricos")
marcas_2021 = pd.read_excel(xls, "marcas2021")

datos = []

# ==============================
# 2. Función expandir
# ==============================

def expandir(df, col_index, year, tipo):
    filas = []
    for _, row in df.iterrows():
        marca = row.iloc[0]
        cantidad = int(row.iloc[col_index])
        filas.extend([[year, marca, tipo]] * cantidad)
    return filas


# ==============================
# 3. 2025–2022 (datos reales)
# ==============================

# 2025-2024
datos += expandir(marcas_2025_h, 1, 2025, "Hibrido")
datos += expandir(marcas_2025_h, 2, 2024, "Hibrido")
datos += expandir(marcas_2025_e, 1, 2025, "Electrico")
datos += expandir(marcas_2025_e, 2, 2024, "Electrico")

# 2023-2022
datos += expandir(marcas_2023_h, 1, 2023, "Hibrido")
datos += expandir(marcas_2023_h, 2, 2022, "Hibrido")
datos += expandir(marcas_2023_e, 1, 2023, "Electrico")
datos += expandir(marcas_2023_e, 2, 2022, "Electrico")


# ==============================
# 4. Separar 2021–2020 (proporcional exacto)
# ==============================

def separar_proporcional(df_marcas, year):

    total_h = int(estado.loc[estado["Año"] == year, "Hibridos"].values[0])
    total_e = int(estado.loc[estado["Año"] == year, "Electricos"].values[0])
    total_general = total_h + total_e

    prop_h = total_h / total_general
    prop_e = total_e / total_general

    col_index = 1 if year == 2021 else 2

    filas = []
    suma_h = 0
    suma_e = 0
    temp = []

    for _, row in df_marcas.iterrows():
        marca = row.iloc[0]
        total_marca = int(row.iloc[col_index])

        h = int(round(total_marca * prop_h))
        e = int(round(total_marca * prop_e))

        suma_h += h
        suma_e += e

        temp.append([marca, h, e])

    # Ajuste fino para que coincida EXACTAMENTE
    temp[0][1] += (total_h - suma_h)
    temp[0][2] += (total_e - suma_e)

    for marca, h, e in temp:
        filas.extend([[year, marca, "Hibrido"]] * h)
        filas.extend([[year, marca, "Electrico"]] * e)

    return filas


datos += separar_proporcional(marcas_2021, 2021)
datos += separar_proporcional(marcas_2021, 2020)


# ==============================
# 5. Calcular proporciones promedio reales 2020–2025
# ==============================

df_actual = pd.DataFrame(datos, columns=["Año","Marca","Tipo"])

proporciones_h = (
    df_actual[df_actual["Tipo"]=="Hibrido"]
    .groupby("Marca")
    .size()
)
proporciones_h = proporciones_h / proporciones_h.sum()

proporciones_e = (
    df_actual[df_actual["Tipo"]=="Electrico"]
    .groupby("Marca")
    .size()
)
proporciones_e = proporciones_e / proporciones_e.sum()


# ==============================
# 6. Generar 2014–2019 exactos
# ==============================

for year in range(2014, 2020):

    total_h = int(estado.loc[estado["Año"] == year, "Hibridos"].values[0])
    total_e = int(estado.loc[estado["Año"] == year, "Electricos"].values[0])

    # Híbridos
    suma = 0
    temp = []
    for marca, prop in proporciones_h.items():
        cantidad = int(round(prop * total_h))
        suma += cantidad
        temp.append([marca, cantidad])

    temp[0][1] += (total_h - suma)

    for marca, cantidad in temp:
        datos.extend([[year, marca, "Hibrido"]] * cantidad)

    # Eléctricos
    suma = 0
    temp = []
    for marca, prop in proporciones_e.items():
        cantidad = int(round(prop * total_e))
        suma += cantidad
        temp.append([marca, cantidad])

    temp[0][1] += (total_e - suma)

    for marca, cantidad in temp:
        datos.extend([[year, marca, "Electrico"]] * cantidad)


# ==============================
# 7. Crear DataFrame final
# ==============================

df_final = pd.DataFrame(datos, columns=["Año","Marca","Tipo"])
df_final = df_final.sort_values(["Año","Marca"]).reset_index(drop=True)

df_final.to_csv("DatosAutomovilesFinal.csv", index=False)


# ==============================
# 8. VALIDACIÓN AUTOMÁTICA
# ==============================

print("\nVALIDACIÓN CONTRA EstadoGeneral2025\n")

for year in estado["Año"]:

    total_h = estado.loc[estado["Año"] == year, "Hibridos"].values[0]
    total_e = estado.loc[estado["Año"] == year, "Electricos"].values[0]

    calc_h = len(df_final[(df_final["Año"]==year) & (df_final["Tipo"]=="Hibrido")])
    calc_e = len(df_final[(df_final["Año"]==year) & (df_final["Tipo"]=="Electrico")])

    print(f"{year} → H: {calc_h}/{total_h} | E: {calc_e}/{total_e}")

print("\nProceso completado correctamente.")



VALIDACIÓN CONTRA EstadoGeneral2025

2025 → H: 67899/67899 | E: 19724/19724
2024 → H: 42668/42668 | E: 9178/9178
2023 → H: 27813/27813 | E: 3677/3677
2022 → H: 24556/24556 | E: 3278/3278
2021 → H: 16366/16366 | E: 1329/1329
2020 → H: 4697/4697 | E: 1307/1307
2019 → H: 2209/2209 | E: 926/926
2018 → H: 539/539 | E: 390/390
2017 → H: 62/62 | E: 136/136
2016 → H: 202/202 | E: 73/73
2015 → H: 193/193 | E: 78/78
2014 → H: 224/224 | E: 20/20

Proceso completado correctamente.
